# XAI vs No XAI Comparison (Quercetin)

Comparing the effect of explainable AI (XAI) guidance on molecule optimization performance for Claude Opus 4.5 on the Quercetin similarity + QED task with two different target thresholds (0.75 and 0.8).

In [1]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [2]:
# Define paths
RESULTS_DIR = Path("../../data/results/similarity_qed_quercetin")

# Define experiment groups: (display_name, folder_name)
GROUPS = [
    ("Claude Opus 4.5 (XAI, 0.8)", "claude-opus-4.5_full_xai_0.8"),
    ("Claude Opus 4.5 (Partial XAI, 0.8)", "claude-opus-4.5_partial_xai_0.8"),
    ("Claude Opus 4.5 (No XAI, 0.8)", "claude-opus-4.5_no_xai_0.8"),
    ("Claude Opus 4.5 (No XAI, no description, 0.8)", "claude-opus-4.5_no_description_0.8"),
    ("REINVENT", "reinvent"),
    ("GraphGA", "graph_ga"),
    ("GP-BO", "gpbo"),
]

## Loading Data

In [3]:
def load_trace(p: Path) -> list:
    """Load trace from JSON file."""
    d = json.loads(p.read_text(encoding="utf-8"))
    return d["trace"] if isinstance(d, dict) and "trace" in d else d


def load_results_as_df(results_dir: Path, groups: list) -> pd.DataFrame:
    """Load all results from JSON trace files into a DataFrame.
    
    Returns DataFrame with columns: model, replicate, iteration, score, smiles
    """
    rows = []
    
    for display_name, folder_name in groups:
        folder_path = results_dir / folder_name
        files = sorted(folder_path.glob("sim_qed_relevant_*.json"))
        
        if not files:
            raise FileNotFoundError(f"No files in {folder_path} matching sim_qed_relevant_*.json")
        
        for file_path in files:
            # Extract replicate (e.g., "rep1", "rep2") from filename
            rep_match = re.search(r'rep(\d+)', file_path.name)
            replicate = f"rep{rep_match.group(1)}" if rep_match else "unknown"
            
            try:
                trace = load_trace(file_path)
                for entry in trace:
                    rows.append({
                        "model": display_name,
                        "replicate": replicate,
                        "iteration": int(entry["iteration"]),
                        "score": float(entry["score"]),
                        "smiles": entry["smiles"],
                    })
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
    
    return pd.DataFrame(rows)


# Load all results
df = load_results_as_df(RESULTS_DIR, GROUPS)

In [4]:
# Filter for 0.8 threshold experiments
BASELINE_MODELS = ["REINVENT", "GraphGA", "GP-BO"]
df_08 = df[df["model"].str.contains("0.8") | df["model"].isin(BASELINE_MODELS)].copy()

# Compute best-so-far (cummax needs data sorted by iteration)
df_08["best_so_far"] = (
    df_08.sort_values("iteration")
    .groupby(["model", "replicate"])["score"]
    .cummax()
)

---
## Iteration to Reach Threshold

In [5]:
def first_iteration_above_threshold(group, threshold):
    """Return first iteration where best_so_far >= threshold, or NaN if never reached."""
    above = group[group["best_so_far"] >= threshold]
    if len(above) > 0:
        return above["iteration"].min()
    return np.nan

In [6]:
# Color palette for explanation conditions + baselines
model_colors_08 = {
    'Claude Opus 4.5 (XAI, 0.8)': '#2E86AB',
    'Claude Opus 4.5 (Partial XAI, 0.8)': '#00a69d',
    'Claude Opus 4.5 (No XAI, 0.8)': '#ffcc00',
    'Claude Opus 4.5 (No XAI, no description, 0.8)': '#f58220',
    'REINVENT': '#A014F5',
    'GraphGA': "#91114F",
    'GP-BO': "#E239EB",
}
legend_order_08 = [
    'Claude Opus 4.5 (XAI, 0.8)',
    'Claude Opus 4.5 (Partial XAI, 0.8)',
    'Claude Opus 4.5 (No XAI, 0.8)',
    'Claude Opus 4.5 (No XAI, no description, 0.8)',
    'REINVENT',
    'GraphGA',
    'GP-BO',
]

In [7]:
# Compute for threshold 0.8
THRESHOLD_08 = 0.8
iter_to_threshold_08 = (
    df_08
    .groupby(["model", "replicate"])
    .apply(lambda g: first_iteration_above_threshold(g, THRESHOLD_08), include_groups=False)
    .reset_index(name=f"iter_to_{THRESHOLD_08}")
)

# Summarize by model with standard deviation
summary_rows_08 = []
for model, group in iter_to_threshold_08.groupby("model"):
    values = group[f"iter_to_{THRESHOLD_08}"]
    mean_val = values.mean()
    std_val = values.std(ddof=1)
    n_total = len(values)
    never_reached = values.isna().sum()
    success_rate = (n_total - never_reached) / n_total * 100
    summary_rows_08.append({
        "Model": model,
        "Mean Iteration": mean_val,
        "Std Dev": std_val,
        "N": n_total,
        "Never Reached": never_reached,
        "Success Rate (%)": success_rate
    })

threshold_summary_08 = pd.DataFrame(summary_rows_08).set_index("Model").round(2)

print(f"Iteration to reach {THRESHOLD_08} threshold (best-so-far) with mean ± SD")
threshold_summary_08

Iteration to reach 0.8 threshold (best-so-far) with mean ± SD


,Mean Iteration,Std Dev,N,Never Reached,Success Rate (%)
Model,,,,,
"Claude Opus 4.5 (No XAI, 0.8)",12.00,7.55,3,0,100.0
"Claude Opus 4.5 (No XAI, no description, 0.8)",NaN,NaN,3,3,0.0
"Claude Opus 4.5 (Partial XAI, 0.8)",5.67,1.53,3,0,100.0
"Claude Opus 4.5 (XAI, 0.8)",5.33,2.52,3,0,100.0
GP-BO,NaN,NaN,3,3,0.0
GraphGA,NaN,NaN,3,3,0.0
REINVENT,NaN,NaN,3,3,0.0
